<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 18</h2>
<h2>Eyler funksiyası, Çin qalıqlar teoremi və sadəlik testləri</h2>
<h3 style="color: #8B4513;">Mühazirə 18 üçün praktiki laboratoriya</h3>
<p><b>Ön şərt:</b> modul arifmetika, ƏBOB, genişləndirilmiş Evklid və sürətli modul qüvvət</p>
<p><b>Müddət:</b> 2 saat · <b>Maksimum:</b> 10 bal</p>
</div>

## 🎯 Məqsədlər

Bu məşğələni bitirdikdən sonra siz:

- Eyler funksiyasını sadə vuruqlara ayırma ilə hesablayacaq və nəticəni birbaşa sayma ilə yoxlayacaqsınız;
- Ferma və Eyler teoremlərini yalnız onların şərtləri ödəndikdə tətbiq edəcəksiniz;
- cüt-cüt qarşılıqlı sadə modullar üçün CRT və Garner rekonstruksiyasını yerinə yetirəcəksiniz;
- RSA deşifrləməsinin CRT formasını birbaşa hesabla müqayisə edəcəksiniz;
- Fermat və Miller–Rabin testlərinin zəmanətlərini və Carmichael ədədlərinin rolunu izah edəcəksiniz;
- AKS-də polinom konqruensiyasının tam alqoritmin yalnız bir mərhələsi olduğunu göstərəcəksiniz.

## 🧰 Mühit

Bu notebook yalnız Python standart kitabxanasından istifadə edir. Şəbəkə, paket quraşdırılması və gizli fallback yoxdur.

In [ ]:
import math
import secrets

print("✅ Python standart kitabxanası hazırdır")

## 1. Eyler funksiyası φ(n) (2 bal)

φ(n), 1-dən n-ə qədər n ilə qarşılıqlı sadə olan tam ədədlərin sayıdır. Əgər
$n = \prod p_i^{e_i}$ isə

$$\varphi(n)=n\prod_{p_i\mid n}\left(1-\frac{1}{p_i}\right).$$

Xüsusi hal: φ(1)=1. Aşağıdakı kod həm faktorizasiya formulunu, həm də kiçik n üçün birbaşa saymanı yoxlayır.

In [ ]:
def prime_factors(n):
    """n >= 1 üçün {sadə_vuruq: qüvvət} qaytarır."""
    if n < 1:
        raise ValueError("n >= 1 olmalıdır")
    factors = {}
    divisor = 2
    while divisor * divisor <= n:
        while n % divisor == 0:
            factors[divisor] = factors.get(divisor, 0) + 1
            n //= divisor
        divisor = 3 if divisor == 2 else divisor + 2
    if n > 1:
        factors[n] = factors.get(n, 0) + 1
    return factors

def euler_phi(n):
    """Sadə vuruqlar formuluna əsasən φ(n)."""
    if n < 1:
        raise ValueError("n >= 1 olmalıdır")
    result = n
    for prime in prime_factors(n):
        result -= result // prime
    return result

def euler_phi_bruteforce(n):
    """Kiçik n üçün tərifdən birbaşa yoxlama."""
    if n < 1:
        raise ValueError("n >= 1 olmalıdır")
    return sum(math.gcd(k, n) == 1 for k in range(1, n + 1))

for n in [1, 2, 9, 10, 36, 100, 561]:
    value = euler_phi(n)
    assert value == euler_phi_bruteforce(n)
    print(f"φ({n}) = {value}; vuruqlar = {prime_factors(n)}")

### ✍️ Çalışma 1

φ(45), φ(64), φ(77) və φ(100)-ü hesablayın. Hər cavabı sadə vuruqlar formulası və birbaşa sayma ilə yoxlayın.

In [ ]:
expected_phi = {45: 24, 64: 32, 77: 60, 100: 40}
for n, expected in expected_phi.items():
    actual = euler_phi(n)
    assert actual == expected == euler_phi_bruteforce(n)
    print(f"✅ φ({n}) = {actual}")

## 2. Ferma və Eyler teoremləri (1 bal)

- p sadə və p ∤ a olduqda $a^{p-1}\equiv1\pmod p$.
- gcd(a,n)=1 olduqda $a^{\varphi(n)}\equiv1\pmod n$.

Şərtlər nəticənin bir hissəsidir: gcd(a,n)≠1 olduqda Eyler teoremini bu formada tətbiq etmək olmaz. Ferma konqruensiyasını keçmək də n-in sadə olduğunu sübut etmir.

In [ ]:
def verify_euler_theorem(a, n):
    if n < 1:
        raise ValueError("n >= 1 olmalıdır")
    if math.gcd(a, n) != 1:
        raise ValueError("Eyler teoremi üçün gcd(a, n) = 1 olmalıdır")
    return pow(a, euler_phi(n), n) == 1 % n

for a, n in [(3, 10), (5, 12), (7, 40), (10, 77)]:
    assert verify_euler_theorem(a, n)
    print(f"✅ {a}^φ({n}) ≡ 1 (mod {n})")

try:
    verify_euler_theorem(6, 15)
except ValueError as error:
    print(f"✅ Şərt yoxlaması: {error}")
else:
    raise AssertionError("gcd şərti pozulduqda xəta gözlənilirdi")

## 3. Çin qalıqlar teoremi (CRT) (2 bal)

Modullar cüt-cüt qarşılıqlı sadədirsə, $x\equiv a_i\pmod{n_i}$ sistemi
$N=\prod n_i$ moduluna görə yeganə həllə malikdir. Implementasiya modulların müsbət və cüt-cüt qarşılıqlı sadə olmasını yoxlayır.

In [ ]:
def extended_gcd(a, b):
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1
    while r:
        quotient = old_r // r
        old_r, r = r, old_r - quotient * r
        old_s, s = s, old_s - quotient * s
        old_t, t = t, old_t - quotient * t
    return old_r, old_s, old_t

def mod_inverse(a, modulus):
    if modulus <= 1:
        raise ValueError("modul > 1 olmalıdır")
    gcd_value, x, _ = extended_gcd(a % modulus, modulus)
    if gcd_value != 1:
        raise ValueError("modul tərs mövcud deyil")
    return x % modulus

def validate_crt_input(residues, moduli):
    if not residues or len(residues) != len(moduli):
        raise ValueError("qalıq və modul siyahıları eyni, sıfırdan böyük uzunluqda olmalıdır")
    if any(modulus <= 1 for modulus in moduli):
        raise ValueError("bütün modullar > 1 olmalıdır")
    for i, left in enumerate(moduli):
        for right in moduli[i + 1:]:
            if math.gcd(left, right) != 1:
                raise ValueError("modullar cüt-cüt qarşılıqlı sadə olmalıdır")

def chinese_remainder(residues, moduli):
    validate_crt_input(residues, moduli)
    total_modulus = math.prod(moduli)
    solution = 0
    for residue, modulus in zip(residues, moduli):
        partial = total_modulus // modulus
        solution += (residue % modulus) * partial * mod_inverse(partial, modulus)
    solution %= total_modulus
    assert all(solution % m == a % m for a, m in zip(residues, moduli))
    return solution, total_modulus

x, modulus = chinese_remainder([2, 3, 2], [3, 5, 7])
assert (x, modulus) == (23, 105)
print(f"✅ x ≡ {x} (mod {modulus})")

### ✍️ Çalışma 2

Sistemi həll edin: $x\equiv1\pmod4$, $x\equiv4\pmod5$, $x\equiv3\pmod6$.

Diqqət: 4 və 6 qarşılıqlı sadə deyil. Əvvəl şərti yoxlayın, sonra ekvivalent cüt-cüt qarşılıqlı sadə sistem qurun.

In [ ]:
try:
    chinese_remainder([1, 4, 3], [4, 5, 6])
except ValueError as error:
    print(f"✅ İlkin sistem standart CRT şərtini ödəmir: {error}")
else:
    raise AssertionError("qarşılıqlı sadə olmayan modullar rədd edilməli idi")

# x ≡ 1 (mod 4) şərti x ≡ 3 (mod 6) ilə birlikdə x ≡ 9 (mod 12) verir.
x, modulus = chinese_remainder([9, 4], [12, 5])
assert (x, modulus) == (9, 60)
assert x % 4 == 1 and x % 5 == 4 and x % 6 == 3
print(f"✅ x ≡ {x} (mod {modulus})")

## 4. Garner rekonstruksiyası (1 bal)

Garner yanaşması həlli qarışıq əsaslı rəqəmlərlə mərhələli qurur. Aşağıdakı forma hər addımda cari həllə yeni konqruensiya əlavə edir və sonda klassik CRT ilə eyni qalığı qaytarmalıdır.

In [ ]:
def garner_reconstruct(residues, moduli):
    validate_crt_input(residues, moduli)
    solution = 0
    product = 1
    digits = []
    for residue, modulus in zip(residues, moduli):
        digit = ((residue - solution) * mod_inverse(product, modulus)) % modulus
        digits.append(digit)
        solution += digit * product
        product *= modulus
    solution %= product
    assert all(solution % m == a % m for a, m in zip(residues, moduli))
    return solution, product, digits

residues = [2, 3, 2]
moduli = [3, 5, 7]
crt_x, crt_modulus = chinese_remainder(residues, moduli)
garner_x, garner_modulus, digits = garner_reconstruct(residues, moduli)
assert (garner_x, garner_modulus) == (crt_x, crt_modulus)
print(f"✅ Garner rəqəmləri = {digits}; x = {garner_x} (mod {garner_modulus})")

## 5. RSA-CRT rekonstruksiyası (1 bal)

Tədris ölçülü RSA nümunəsində $d_p=d\bmod(p-1)$, $d_q=d\bmod(q-1)$ və
$q_{inv}=q^{-1}\bmod p$ saxlanılır. Bu parametrlər gizli açarın hissəsidir; p və q-nu silib CRT formasını saxlamaq onları «yox etmək» demək deyil.

In [ ]:
def rsa_crt_decrypt(ciphertext, p, q, d):
    """Tədris nümunəsi; caller p və q-nun sadəliyini ayrıca yoxlamalıdır."""
    if p <= 2 or q <= 2 or p == q or p % 2 == 0 or q % 2 == 0:
        raise ValueError("p və q fərqli tək ədədlər olmalıdır")
    n = p * q
    dp = d % (p - 1)
    dq = d % (q - 1)
    q_inv = mod_inverse(q, p)
    message_p = pow(ciphertext, dp, p)
    message_q = pow(ciphertext, dq, q)
    h = (q_inv * (message_p - message_q)) % p
    message = (message_q + h * q) % n
    return message, {"dp": dp, "dq": dq, "q_inv": q_inv}

p, q, e = 61, 53, 17
n = p * q
d = mod_inverse(e, (p - 1) * (q - 1))
message = 65
ciphertext = pow(message, e, n)
direct = pow(ciphertext, d, n)
via_crt, private_crt = rsa_crt_decrypt(ciphertext, p, q, d)
assert direct == via_crt == message
print(f"✅ Birbaşa = CRT = {via_crt}; gizli CRT parametrləri: {private_crt}")

## 6. Sadəlik testləri (3 bal)

Trial division kiçik ədədlər üçün deterministik oracle verir. Fermat testi yalnız zəruri şərti yoxlayır və Carmichael ədədlərini qəbul edə bilər. Miller–Rabin strong probable-prime testidir: tək mürəkkəb n üçün bazaların ən çox dörddə biri strong liar olur; müstəqil və bərabər paylanmış k təsadüfi baza ilə false-positive ehtimalı ən çox $(1/4)^k$-dir.

In [ ]:
def is_prime_trial(n):
    """Kiçik n üçün deterministik trial division."""
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    divisor = 3
    while divisor * divisor <= n:
        if n % divisor == 0:
            return False
        divisor += 2
    return True

known_primes = [2, 3, 5, 17, 97, 997]
known_composites = [-1, 0, 1, 4, 9, 21, 221, 561, 1105]
assert all(is_prime_trial(n) for n in known_primes)
assert not any(is_prime_trial(n) for n in known_composites)
print("✅ Trial division testləri keçdi")

### 6.1 Fermat testi və Carmichael ədədləri

561=3·11·17 Carmichael ədədidir. gcd(a,561)=1 olan hər a üçün $a^{560}\equiv1\pmod{561}$, buna görə tək Fermat testi onu sadəlik sübutu kimi ayıra bilmir.

In [ ]:
def fermat_passes(n, base):
    if n <= 2 or not 1 < base < n:
        raise ValueError("n > 2 və 1 < base < n olmalıdır")
    if math.gcd(base, n) != 1:
        return False
    return pow(base, n - 1, n) == 1

for base in [2, 50, 101]:
    assert math.gcd(base, 561) == 1
    assert fermat_passes(561, base)
    print(f"⚠️ 561 Fermat bazası {base} üçün testi keçir")
assert not is_prime_trial(561)

### 6.2 Miller–Rabin

$n-1=2^s d$ (d tək) yazılır. Hər baza üçün $a^d$ və ardıcıl kvadratlar yoxlanır. Sabit bazalar yalnız göstərilən/test edilmiş diapazonda deterministik nəticə verir; ümumi böyük ədədlər üçün bazalar müstəqil, bərabər paylanmış şəkildə seçilir.

In [ ]:
def miller_rabin_with_bases(n, bases):
    """Verilmiş bazalar üçün strong probable-prime yoxlaması."""
    bases = list(bases)
    if not bases:
        raise ValueError("ən azı bir baza verilməlidir")
    if n < 2:
        return False
    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    for prime in small_primes:
        if n % prime == 0:
            return n == prime

    d = n - 1
    s = 0
    while d % 2 == 0:
        s += 1
        d //= 2

    for base in bases:
        a = base % n
        if a in (0, 1):
            continue
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = (x * x) % n
            if x == n - 1:
                break
        else:
            return False
    return True

def is_probable_prime(n, rounds=32):
    """Böyük n üçün təsadüfi bazalı Miller–Rabin; probable-prime nəticəsi verir."""
    if rounds < 1:
        raise ValueError("rounds >= 1 olmalıdır")
    if n < 5:
        return n in (2, 3)
    if n % 2 == 0:
        return False
    bases = [2 + secrets.randbelow(n - 3) for _ in range(rounds)]
    return miller_rabin_with_bases(n, bases)

teaching_bases = [2, 3, 5, 7, 11]
for n in range(2, 5_000):
    assert miller_rabin_with_bases(n, teaching_bases) == is_prime_trial(n)
print("✅ 2...4999 diapazonunda nəticələr trial division ilə üst-üstə düşdü")

for n in [97, 997, 65_537]:
    assert is_probable_prime(n, rounds=16)  # sadə n hər düzgün bazanı keçir
for n in [221, 561, 1105, 1729]:
    assert not miller_rabin_with_bases(n, teaching_bases)
print("✅ Sadə və mürəkkəb nümunələr keçdi")

## 7. AKS-in düzgün sərhədi

AKS deterministik polinom-zamanlı primality alqoritmidir, lakin aşağıdakı polinom konqruensiyası təkbaşına tam test deyil:

$$(X+a)^n\equiv X^n+a\pmod{X^r-1,n}.$$

Tam sxem ən azı bu mərhələləri birlikdə istifadə edir:

1. n-in $a^b$ formasında perfect power olub-olmadığını yoxlamaq;
2. $\operatorname{ord}_r(n)>(\log_2 n)^2$ olan r tapmaq;
3. $1<\gcd(a,n)<n$ verən kiçik a-ları yoxlamaq;
4. n≤r halını ayrıca emal etmək;
5. tələb olunan məhdud a diapazonunda polinom konqruensiyalarını yoxlamaq.

Bu notebook AKS-i qısaldılmış kodla «implementasiya etmir»: məqsəd mərhələləri və zəmanəti Miller–Rabin-dən ayırmaqdır. Praktik vaxt haqqında platformasız universal rəqəm verilmir.

In [ ]:
# Bütün əsas bloklar üçün inteqrasiya və sərhəd testləri
for n in range(1, 301):
    assert euler_phi(n) == euler_phi_bruteforce(n)

crt_cases = [
    ([2, 3, 2], [3, 5, 7]),
    ([1, 4], [12, 5]),
    ([-1, 8, 15], [5, 7, 11]),
]
for residues, moduli in crt_cases:
    crt_value, crt_modulus = chinese_remainder(residues, moduli)
    garner_value, garner_modulus, _ = garner_reconstruct(residues, moduli)
    assert (crt_value, crt_modulus) == (garner_value, garner_modulus)
    assert all(crt_value % m == a % m for a, m in zip(residues, moduli))

for n in range(2, 5_000):
    assert miller_rabin_with_bases(n, [2, 3, 5, 7, 11]) == is_prime_trial(n)

print("✅ İnteqrasiya testləri: φ(1...300), CRT/Garner və primality(2...4999)")

## 🏠 Ev tapşırığı

1. φ(1...1000) üçün formula və brute-force nəticələrini müqayisə edən test yazın; runtime fərqini təkrarlanan ölçmə ilə izah edin.
2. Cüt-cüt qarşılıqlı sadə olmayan, amma həlli olan ümumiləşdirilmiş CRT sistemi üçün ayrıca funksiya yazın və həll şərtini sübut edin.
3. RSA-CRT hesabında $m_p$ və ya $m_q$ nəticəsinin bir bitini dəyişdirin. Səhv nəticə ilə düzgün nəticənin fərqindən faktorun necə tapıldığını kiçik nümunədə göstərin; bunu production fault attack modeli kimi təqdim etməyin.
4. 561, 1105 və 1729 üçün Fermat və Miller–Rabin nəticələrini müqayisə edin.
5. AKS alqoritminin hər mərhələsinin məqsədini yazın və yalnız polinom konqruensiyasını yoxlamağın niyə kifayət etmədiyini izah edin.

## 📌 Yekun

- φ(n) faktorizasiya ilə səmərəli hesablanır və teoremlərin şərtləri ayrıca yoxlanır.
- Standart CRT implementasiyası burada yalnız cüt-cüt qarşılıqlı sadə modullar üçündür; Garner eyni həlli mərhələli qurur.
- RSA-CRT sürətləndirmə ilə yanaşı gizli CRT parametrləri və fault-check tələbləri yaradır.
- Fermat testini keçmək sadəlik sübutu deyil; Carmichael ədədləri əsas əks nümunədir.
- Miller–Rabin probable-prime nəticəsi və seçilmiş baza modelinə bağlı səhv həddi verir.
- AKS deterministikdir, amma polinom konqruensiyası tam alqoritmin yalnız bir hissəsidir.

### Mənbələr

- [Agrawal, Kayal, Saxena — PRIMES is in P](https://www.microsoft.com/en-us/research/publication/primes-is-in-p/) — tam AKS alqoritmi və teorem.
- Menezes, van Oorschot, Vanstone — Handbook of Applied Cryptography, fəsillər 2 və 4.
- [RFC 8017](https://www.rfc-editor.org/rfc/rfc8017) — RSA və CRT private-key representation.

<div style="background-color: #f0f0f0; padding: 16px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 18 tamamlandı!</h2>
<p>Notebook-un pass olması bütün code cells-in gizli skip olmadan icrasını və assertions-in keçməsini tələb edir.</p>
</div>